# Chest X-ray progression: experiments A-F

Compare seven models on the same patient splits. Each image model uses the **prior and current X-rays**; the prediction is **improving, stable, or worsening** for each of five diseases.

| ID | Encoder / initialization | Clinical features | Question |
|:--|:--|:--|:--|
| **A** | DenseNet-121, ImageNet | None | Image-only baseline |
| **A2** | DenseNet-121, diagnosis checkpoint | None | Does diagnosis pretraining help? |
| **B** | DenseNet-121, ImageNet | Matched | Do clinical features improve A? |
| **C** | RAD-DINO | None | Does the new encoder improve image-only prediction? |
| **D** | RAD-DINO | Matched | Do clinical features improve C? |
| **E** | RAD-DINO | Shuffled within each split | Does correct feature matching matter? |
| **F** | Tabular neural network | Matched; no images | How much can clinical features predict alone? |

**Workflow:** setup -> reviewed dataset -> clinical features -> A2 checkpoint -> A-F -> results.

Run from top to bottom. The default settings reopen the latest run and load its completed experiments. Set `RESUME_RUN=False` for a new training run. Optional encoder screening, logistic regression, and archive extraction are grouped at the end.

A pair can have several disease labels. `-100` masks a disease without a usable progression label. Edema and Pulmonary edema overlap under the current labeling rules.


## 1. Get ready
Use the `longitudinal_progression` Python environment. Run the cells in order. Each heading describes the next small step.


### 1.1 Load Python libraries

These libraries read tables, draw charts, and run the models.


In [ ]:
from datetime import datetime
from pathlib import Path
import gc
import json
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Markdown, display


### 1.2 Find this project

Check the project folder and make its Python modules available.


In [ ]:
PROJECT_DIR = Path(os.environ.get("BN5212_PROJECT_DIR", Path.cwd())).resolve()
if not (PROJECT_DIR / "longitudinal_ext").is_dir():
    raise FileNotFoundError("Open this notebook from the midterm_prognosis project folder.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
torch.set_num_threads(4)


### 1.3 Find downloaded model weights

Reuse the pretrained model files already downloaded on this machine.


In [ ]:
# Use the pretrained models already downloaded for the local run.
MODEL_CACHE = PROJECT_DIR / "extensions_outputs" / "model_cache"
os.environ.setdefault("HF_HOME", str(MODEL_CACHE / "huggingface"))
os.environ.setdefault("TORCH_HOME", str(MODEL_CACHE / "torch"))
os.environ.setdefault("XRV_CACHE_DIR", str(MODEL_CACHE / "xrv"))


### 1.4 Load project functions

The project functions handle data preparation, model training, and result summaries.


In [ ]:
from longitudinal_ext import CLASS_NAMES, DISEASES
from longitudinal_ext.mimic_iv_features import build_pair_features, coverage_report
from longitudinal_ext.pair_data import detect_image_columns
from longitudinal_ext.reporting import collect, summary, plot_confusions
from local_notebook_helpers import load_reviewed_data, run_or_load
from audit_mimic_features import audit_features


### 1.5 Locate the source data

These paths identify the X-rays, clinical records, original labels, accepted reviews, and saved patient groups.


In [ ]:
DATA_ROOT = PROJECT_DIR.parent
REPORTS_ROOT = DATA_ROOT / "MIMIC-CXR" / "p10" / "p10"
MIMIC_IV_ROOT = DATA_ROOT / "MIMIC-IV" / "3.1"
LABELLING_DIR = PROJECT_DIR / "labelling"
HYBRID_DIR = LABELLING_DIR / "prognosis_labels_hybrid"
DIAGNOSIS_DIR = LABELLING_DIR / "cxr_labels_selected_everpositive"

BASE_PAIR_CSV = HYBRID_DIR / "all_disease_pairs.csv"
PROGNOSIS_REVIEWS_PATH = HYBRID_DIR / "reviewed_only_excluding_7_conflicts.csv"
SPLIT_CSV = LABELLING_DIR / "prognosis_patient_splits.csv"


### 1.6 Choose what to run

Keep `RESUME_RUN=True` to reopen the latest run. `REUSE_COMPLETED=True` loads matching saved results. The optional analyses are off by default.


In [ ]:
RESUME_RUN = True
REUSE_COMPLETED = True
RUN_EXPERIMENTS = True
RUN_BACKBONE = False                # Optional preparation for A2.
RUN_ENCODER_SCREEN = False          # Optional supporting analysis.
RUN_TAB_BASELINE = False            # Optional logistic-regression baseline.
REBUILD_TAB = False


### 1.7 Choose the results folder

Each run has its own folder for data, model files, and results. A new folder is created when you start a new run.


### 1.8 Name the prepared data files

These are the reviewed image-pair table, clinical feature table, and stored copies of decoded images.


In [ ]:
PAIR_CSV = HYBRID_DIR / "all_disease_pairs.csv"
PROGNOSIS_REVIEWS_PATH = HYBRID_DIR / "reviewed_only_excluding_7_conflicts.csv"
SPLIT_CSV = LABELLING_DIR / "prognosis_patient_splits.csv"

RESUME_RUN = True
REUSE_COMPLETED = True
RUN_EXPERIMENTS = True
RUN_BACKBONE = False                # Optional preparation for A2.
RUN_ENCODER_SCREEN = False          # Optional supporting analysis.
RUN_TAB_BASELINE = False            # Optional logistic-regression baseline.
REBUILD_TAB = False

latest_file = PROJECT_DIR / "training_outputs" / "latest_reviewed_run.txt"
saved_run = Path(latest_file.read_text().strip()) if latest_file.exists() else None
new_run_dir = PROJECT_DIR / "training_outputs" / datetime.now().strftime("reviewed_local_%Y%m%d_%H%M%S")
default_run = saved_run if RESUME_RUN and saved_run is not None else new_run_dir
RUN_DIR = Path(os.environ.get("BN5212_RUN_DIR", str(default_run)))
RUN_DIR.mkdir(parents=True, exist_ok=True)

PAIR_CSV = RUN_DIR / "data" / "all_disease_pairs_reviewed.csv"
TAB_CSV = RUN_DIR / "features" / "pair_tabular_features.csv"
IMAGE_CACHE = PROJECT_DIR / "extensions_outputs" / "image_cache"


### 1.9 Set the training options

An epoch is one pass through the training data. Batch size is the number of image pairs processed together. A seed controls random choices so a run can be repeated.


In [ ]:
EPOCHS = 20
EXT_SEEDS = [42]
PAIR_BATCH_SIZE = 8
BACKBONE_LR = 1e-5
HEAD_LR = 1e-4
FINAL_LR_RATIO = 0.1
BASE_ENCODER = "densenet121_imagenet"
NEW_ENCODER = "hf:microsoft/rad-dino"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


### 1.10 Select the experiments

A, A2, and B-F are the seven comparisons listed at the top. They use the same data and training settings.


In [ ]:
EXT_TO_RUN = [
    "A_base_img", "A2_base_diag_img", "B_base_img_tab", "C_new_img",
    "D_new_img_tab", "E_new_img_tab_shuffled", "F_tab_nn",
]
print(f"Device: {DEVICE} | epochs: {EPOCHS} | seeds: {EXT_SEEDS}")
print("Run folder:", RUN_DIR)


## 2. Add the reviewed disease labels

Two different things are counted here:

- **One X-ray pair:** an earlier and a later X-ray from the same patient.
- **One disease label:** whether one disease improved, stayed stable, or worsened between those X-rays.

For example, one pair can have **Pleural effusion: improving** and **Cardiomegaly: stable**. That is **one pair with two labels**.

The accepted review file excludes the seven conflicting progression decisions and the `no_comparison` decisions. Patients keep their original training, validation, or test group.


### 2.1 Check that the input files exist

Stop here if an original data file, accepted review file, or patient-group file is missing.


In [ ]:
required_files = [BASE_PAIR_CSV, PROGNOSIS_REVIEWS_PATH, SPLIT_CSV]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing source files: " + ", ".join(missing))


### 2.2 Load the reviewed dataset

Add accepted decisions to the original dataset, or reuse the saved version when the input files have not changed.


In [ ]:
pair_df, review_merge_summary = load_reviewed_data(
    base_csv=BASE_PAIR_CSV,
    reviews_csv=PROGNOSIS_REVIEWS_PATH,
    split_csv=SPLIT_CSV,
    reports_root=REPORTS_ROOT,
    output_csv=PAIR_CSV,
)


### 2.3 Check the X-ray files

Every pair needs both image files. This checks that they are available before training.


In [ ]:
image_columns = ["prior_dicom_path", "current_dicom_path"]
missing_images = [path for column in image_columns for path in pair_df[column].unique()
                  if not Path(path).is_file()]
if missing_images:
    raise FileNotFoundError(f"{len(missing_images)} image paths are unavailable. First: {missing_images[0]}")


### 2.4 How many X-ray pairs are there?

**915 original pairs + 22 added pairs = 937 pairs in the updated dataset.** A pair is counted once even when it has labels for several diseases.


In [ ]:
pair_counts = pd.Series({
    "X-ray pairs in the original dataset": review_merge_summary["base_pairs"],
    "Additional X-ray pairs from the reviews": review_merge_summary["added_pairs"],
    "X-ray pairs in the updated dataset": review_merge_summary["pairs"],
}, name="Number of X-ray pairs")
display(pair_counts.to_frame())


### 2.5 What did the 307 reviews add?

**307 reviewed disease decisions = 50 added labels + 257 confirmations.** The 50 added labels cover diseases that previously had no usable label, including labels on the new pairs. The 257 confirmations agree with existing labels and are not added again. These are disease-label counts, so they are different from the 22 added image pairs.


In [ ]:
review_counts = pd.DataFrame([
    {"What is counted": "All accepted disease decisions", "Count": review_merge_summary["reviewed_labels"],
     "Meaning": "One review for one disease on one X-ray pair."},
    {"What is counted": "Additional usable disease labels", "Count": review_merge_summary["new_labels"],
     "Meaning": "The dataset previously had no usable label for that disease and pair."},
    {"What is counted": "Existing disease labels confirmed", "Count": review_merge_summary["confirmed_labels"],
     "Meaning": "The review agrees with a label already in the dataset."},
]).set_index("What is counted")
with pd.option_context("display.max_colwidth", None):
    display(review_counts)


### 2.6 How are the patients divided?

**Training** data teach the model. **Validation** data help choose the model. **Test** data measure its final performance. All pairs from one patient stay in the same group.


In [ ]:
patient_groups = pair_df.groupby("split").agg(
    **{"X-ray pairs": ("subject_id", "size"), "Patients": ("subject_id", "nunique")}
).rename(index={"train": "Training", "val": "Validation", "test": "Test"})
patient_groups.index.name = "Patient group"
display(patient_groups)


### 2.7 Count labels for each disease

These are the labels actually used by the progression models. A label is **improving**, **stable**, or **worsening**. A value of `-100` means there is no usable label for that disease; the model ignores it. Diagnosis terms such as `uncertain` are not progression targets.


In [ ]:
label_rows = []
for split, group in pair_df.groupby("split"):
    for disease in DISEASES:
        counts = group[disease].value_counts()
        label_rows.append({"split": split, "disease": disease,
                           **{name: int(counts.get(i, 0)) for i, name in enumerate(CLASS_NAMES)},
                           "masked": int(counts.get(-100, 0))})
label_table = pd.DataFrame(label_rows).rename(columns={
    "split": "Patient group", "disease": "Disease", "masked": "No usable label",
})
label_table["Patient group"] = label_table["Patient group"].replace(
    {"train": "Training", "val": "Validation", "test": "Test"}
)
display(label_table.set_index(["Patient group", "Disease"]))


### 2.8 Show the label counts in a chart

This chart combines training, validation, and test labels. It counts disease labels, not patients or individual image files.


In [ ]:
plot_counts = pd.DataFrame({d: [(pair_df[d] == i).sum() for i in range(3)]
                            for d in DISEASES}, index=CLASS_NAMES).T
ax = plot_counts.plot.bar(figsize=(11, 4), color=["#009E73", "#0072B2", "#D55E00"], rot=15)
for bars in ax.containers:
    ax.bar_label(bars, padding=3, fontsize=9)
ax.set(title="Progression labels across all splits", xlabel="Disease", ylabel="Labeled pairs")
ax.margins(y=0.18)
plt.tight_layout()
plt.show()


## 3. Prepare the clinical measurements
B, D, E, and F use the same MIMIC-IV clinical data. These include measurements near the X-ray visit, changes since the earlier visit, and patient information. E deliberately gives each pair another pair's clinical data within the same patient group.


### 3.1 Build or reuse the clinical table

Reuse the saved table when it exists. Set `REBUILD_TAB=True` to calculate it again from the source clinical records.


In [ ]:
IMG_IN, IMG_TG = detect_image_columns(pair_df)
TAB_CSV.parent.mkdir(parents=True, exist_ok=True)
if REBUILD_TAB or not TAB_CSV.exists():
    if not (MIMIC_IV_ROOT / "hosp").is_dir():
        raise FileNotFoundError(f"MIMIC-IV tables are required under {MIMIC_IV_ROOT}.")
    build_pair_features(pair_df, IMG_IN, IMG_TG, MIMIC_IV_ROOT,
                        TAB_CSV.parent / "mimic_cache", TAB_CSV)


### 3.2 Match clinical records to image pairs

Check that the clinical table contains exactly the same patients and study pairs as the reviewed image dataset.


In [ ]:
tab_features = pd.read_csv(TAB_CSV, dtype={key: str for key in
                           ["subject_id", "input_study_id", "target_study_id"]})
keys = ["subject_id", "input_study_id", "target_study_id"]
expected = set(map(tuple, pair_df[keys].astype(str).to_numpy()))
actual = set(map(tuple, tab_features[keys].to_numpy()))
if expected != actual or len(tab_features) != len(pair_df):
    raise ValueError("The clinical feature table does not match the reviewed dataset.")


### 3.3 See which measurements are available

Each value is a fraction from 0 to 1. `cur` means a measurement is available near the later X-ray; `prev` means near the earlier X-ray; `delta` means both visits have a measurement, so their change can be calculated. Availability does not tell us how useful a measurement is.


In [ ]:
print(f"Clinical features available for all {len(tab_features)} pairs.")
display(coverage_report(tab_features, pair_df))


### 3.4 See which clinical inputs the model uses

Measurements are kept when enough training examples have usable, varying values. Extra flags tell the model when a measurement is missing. These choices use training data only.


In [ ]:
feature_audit, selected_inputs = audit_features(PAIR_CSV, TAB_CSV, TAB_CSV.parent)
display(feature_audit)
print("Retained measurement values:", len(selected_inputs["retained_features"]))
print("Flags marking missing measurements:", len(selected_inputs["missing_indicators"]))
print("Total clinical inputs to the model:", selected_inputs["n_tab"])


## 4. Choose the saved model for A2
A2 starts from a DenseNet model previously trained to recognize whether a disease is mentioned as present, absent, or uncertain in a report. Its diagnosis classes are `not_mentioned`, `uncertain`, `negative`, and `positive`.

A2 then learns a new prediction task with **improving, stable, and worsening** as its outputs. The saved diagnosis model is only its starting point.


### 4.1 Find the saved diagnosis model

Use a completed diagnosis model from this run, or the existing fallback model.


In [ ]:
checkpoint_roots = [RUN_DIR] + ([saved_run] if saved_run is not None and saved_run != RUN_DIR else [])
checkpoints = [folder / "best_backbone.pt" for root in checkpoint_roots
               for folder in root.glob("backbone_*")
               if (folder / "test_metrics.csv").is_file() and (folder / "best_backbone.pt").is_file()]
fallback = PROJECT_DIR / "training_outputs" / "backbone_20261003_150057" / "best_backbone.pt"
CHECKPOINT_PATH = max(checkpoints, key=lambda p: p.stat().st_mtime) if checkpoints else fallback


### 4.2 Optional: train a new diagnosis model

This step runs only when `RUN_BACKBONE=True`. Leave it off to use the saved model and go straight to the seven progression experiments.


In [ ]:
if RUN_BACKBONE:
    from data_loader.cxr_backbone_dataloader import make_backbone_dataloaders
    from model.cxr_backbone import CXRBackbone
    from model.train_backbone import train_backbone

    loaders = make_backbone_dataloaders(
        labels_csv=DIAGNOSIS_DIR / "study_labels.csv", reviews_csv=DIAGNOSIS_DIR / "reviewed_only.csv",
        report_summary_csv=DIAGNOSIS_DIR / "report_summary.csv", split_csv=SPLIT_CSV,
        reports_root=REPORTS_ROOT, batch_size=16, num_workers=0,
        manifest_path=RUN_DIR / "data" / "backbone_image_manifest.csv",
    )
    torch.manual_seed(EXT_SEEDS[0])
    model = CXRBackbone(pretrained=True)
    result = train_backbone(model, loaders, RUN_DIR / datetime.now().strftime("backbone_%Y%m%d_%H%M%S"),
                            epochs=EPOCHS, learning_rate=1e-4, seed=EXT_SEEDS[0], device=DEVICE)
    CHECKPOINT_PATH = result["best_path"]
    model.cpu()
    del model, loaders
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


### 4.3 Confirm the starting model for A2

Check that the selected model file exists before running A2.


In [ ]:
if "A2_base_diag_img" in EXT_TO_RUN and not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError("A2 requires a diagnosis checkpoint. Set CHECKPOINT_PATH or enable RUN_BACKBONE.")
print("A2 initialization:", CHECKPOINT_PATH)


## 5. Run the seven progression experiments
Each image model compares the earlier and later X-rays. Its image representation is `encoder(current) - encoder(prior)`: the difference between the two images' learned features.

B and D combine that image difference with clinical inputs. E uses shuffled clinical inputs. F uses clinical inputs alone.


### 5.1 Define what each experiment receives

These definitions specify the image model, starting weights, and whether clinical inputs are included.


In [ ]:
EXT_EXPERIMENTS = {
    "A_base_img":             dict(encoder=BASE_ENCODER),
    "A2_base_diag_img":       dict(encoder=BASE_ENCODER, diagnosis_checkpoint=CHECKPOINT_PATH),
    "B_base_img_tab":         dict(encoder=BASE_ENCODER, tab_csv=TAB_CSV),
    "C_new_img":              dict(encoder=NEW_ENCODER),
    "D_new_img_tab":          dict(encoder=NEW_ENCODER, tab_csv=TAB_CSV),
    "E_new_img_tab_shuffled": dict(encoder=NEW_ENCODER, tab_csv=TAB_CSV, tab_control="shuffle"),
    "F_tab_nn":               dict(use_images=False, tab_csv=TAB_CSV),
}


### 5.2 Load completed results or train missing models

Experiments run one at a time on the local GPU. When `REUSE_COMPLETED=True`, saved results with matching settings are loaded. Missing models train for the selected number of epochs. Each model keeps the version with the lowest validation loss.


In [ ]:
ext_runs = {}
for name in EXT_TO_RUN:
    ext_runs[name] = []
    for seed in EXT_SEEDS:
        if RUN_EXPERIMENTS:
            result = run_or_load(
                name, reuse=REUSE_COMPLETED, manifest_csv=PAIR_CSV, out_root=RUN_DIR,
                epochs=EPOCHS, batch_size=PAIR_BATCH_SIZE, seed=seed, device=DEVICE,
                backbone_lr=BACKBONE_LR, head_lr=HEAD_LR, final_lr_ratio=FINAL_LR_RATIO,
                image_cache=IMAGE_CACHE, **EXT_EXPERIMENTS[name],
            )
            ext_runs[name].append(result["out_dir"])
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    if not ext_runs[name]:
        del ext_runs[name]
if not ext_runs:
    print("Enable RUN_EXPERIMENTS to load or train the selected experiments.")


## 6. Read the progression results
Each disease is scored on **improving, stable, and worsening**. Macro-F1 averages the F1 scores of these three classes, giving each class equal weight. `mean_over_diseases` then averages the five disease scores. Higher values are better.


### 6.1 Gather and save the scores

Collect the results of A, A2, and B-F into summary tables.


In [ ]:
if ext_runs:
    all_ext = collect(ext_runs)
    validation_summary = summary(all_ext, split="val")
    test_summary = summary(all_ext, split="test")
    all_ext.to_csv(RUN_DIR / "extension_metrics.csv", index=False)
    validation_summary.to_csv(RUN_DIR / "extension_validation_summary.csv")
    test_summary.to_csv(RUN_DIR / "extension_test_summary.csv")


### 6.2 Validation: choose the experiment

Use these scores to choose among experiments. The validation data help make model choices.


In [ ]:
if ext_runs:
    hidden_columns = ["sd_over_seeds"] if len(EXT_SEEDS) == 1 else []
    display(validation_summary.drop(columns=hidden_columns).rename(columns={
        "mean_over_diseases": "Average score across diseases",
        "sd_over_seeds": "Variation across training seeds",
    }))
    print("Highest average validation score:", validation_summary["mean_over_diseases"].idxmax())


### 6.3 Test: measure the chosen model

The test group measures performance on held-out patients. The highest observed test score is not the rule for selecting a model.


In [ ]:
if ext_runs:
    hidden_columns = ["sd_over_seeds"] if len(EXT_SEEDS) == 1 else []
    display(test_summary.drop(columns=hidden_columns).rename(columns={
        "mean_over_diseases": "Average score across diseases",
        "sd_over_seeds": "Variation across training seeds",
    }))


### 6.4 Do the clinical inputs help?

Compare **B minus A** for DenseNet and **D minus C** for RAD-DINO. **D minus E** asks whether matching the clinical data to the correct pair is helpful. A positive difference favors matched clinical inputs; a negative difference favors the comparison model.


In [ ]:
if ext_runs:
    mean_scores = all_ext.groupby(["experiment", "split"])["macro_f1"].mean().unstack()
    contrast_rows = []
    for label, left, right in [
        ("DenseNet: matched tabular minus image only", "B_base_img_tab", "A_base_img"),
        ("RAD-DINO: matched tabular minus image only", "D_new_img_tab", "C_new_img"),
        ("RAD-DINO: matched tabular minus shuffled", "D_new_img_tab", "E_new_img_tab_shuffled"),
    ]:
        if left in mean_scores.index and right in mean_scores.index:
            contrast_rows.append({"comparison": label,
                                  "validation difference": mean_scores.loc[left, "val"] - mean_scores.loc[right, "val"],
                                  "test difference": mean_scores.loc[left, "test"] - mean_scores.loc[right, "test"]})
    display(pd.DataFrame(contrast_rows).set_index("comparison").round(3))


### 6.5 See which progression classes are confused

Rows are the actual labels; columns are the model predictions. Each square shows the number of cases and the percentage within that actual-label row.


In [ ]:
for name, folders in ext_runs.items():
    display(Markdown(f"### {name}"))
    plot_confusions(folders[0], split="test")
    plt.show()


## 7. Compare results before and after the reviews
The updated dataset has **937 pairs and 1,500 disease labels**, compared with **915 pairs and 1,450 labels** before the review merge. Of the 50 added labels, 36 are in training, 4 in validation, and 10 in test.

The comparison also measures the new models on the **original evaluation cases**. This helps distinguish changes in predictions from changes in which cases are evaluated.


### 7.1 Compare the old and updated scores

The original-case columns use the same evaluation cases as before the reviews were added.


In [ ]:
comparison_path = RUN_DIR / "extension_before_after.csv"
if comparison_path.exists():
    before_after = pd.read_csv(comparison_path, index_col="experiment")
    display(before_after.rename(columns={
        "old_val": "Old validation", "new_val": "New validation",
        "old_test": "Old test", "new_test": "New test",
        "new_val_original_cases": "New validation, original cases",
        "new_test_original_cases": "New test, original cases",
    }).round(3))
else:
    print("Complete all seven experiments and run summarize_reviewed_extensions.py to create this table.")


### 7.2 See how uncertain the clinical-feature differences are

The interval gives a range of plausible score differences when test patients are resampled. If the range includes zero, these results do not clearly establish which model is better. This measures patient-sampling uncertainty; it does not measure variation across training seeds.


In [ ]:
intervals_path = RUN_DIR / "extension_paired_comparisons.csv"
if intervals_path.exists():
    display(pd.read_csv(intervals_path).rename(columns={
        "comparison": "Models compared", "test_difference": "Difference in test score",
        "ci_95_low": "Lower end of 95% interval", "ci_95_high": "Upper end of 95% interval",
    }).round(3))
report_path = RUN_DIR / "extension_comparison_report.md"
if report_path.exists():
    display(Markdown(f"[Read the detailed comparison report]({report_path.as_posix()})"))


### Recorded interpretation: 3 November 2026

For `reviewed_local_20261102_191222`, seed 42:

**The clinical-feature conclusion remains uncertain.** Matched RAD-DINO fusion D scores 0.423 on test, versus 0.415 for image-only C and 0.432 for shuffled control E. The paired patient-bootstrap intervals for these differences include zero. DenseNet fusion B also scores below image-only A on test.

A2 has the highest validation macro-F1 (0.463). E has the highest observed test macro-F1 (0.432); model choice should follow validation. These are one-seed results. The local feature reconstruction, training order, A2's newly trained checkpoint, and added evaluation labels mean changes cannot be attributed solely to the added training labels.

Clinical features include order/care-process proxies and have timing limitations. Review notes include assumed or unavailable comparison references. See [the feature audit](mimic_feature_validation.md) and the per-run manual-review audit for details.


## 8. Optional additional comparisons
These supporting analyses are separate from A, A2, and B-F. Saved results are displayed when available. The flags in Section 1 control whether missing analyses are run.


### 8.1 Choose image models for a quick comparison

This comparison keeps each image model fixed and trains a simpler classifier on its image features. Use validation scores to compare these screens. C, D, and E keep RAD-DINO to match the original seven-experiment design.


In [ ]:
from local_notebook_helpers import completed_run

ENCODER_CANDIDATES = [
    BASE_ENCODER, "xrv:densenet121-res224-chex", "xrv:densenet121-res224-nih",
    NEW_ENCODER, "biomedclip", "timm:convnext_small.fb_in22k_ft_in1k",
]


### 8.2 Load or calculate the image-model comparisons

Reuse completed screens. Calculate a missing screen only when `RUN_ENCODER_SCREEN=True`.


In [ ]:
screen_runs = {}
for encoder in ENCODER_CANDIDATES:
    name = "screen_" + encoder.replace(":", "_").replace("/", "_")
    saved = completed_run(RUN_DIR, name, EXT_SEEDS[0], expected={"encoder": encoder})
    if saved is not None:
        screen_runs[encoder] = [saved]
    elif RUN_ENCODER_SCREEN:
        result = run_or_load(name, reuse=REUSE_COMPLETED, manifest_csv=PAIR_CSV, out_root=RUN_DIR,
                             encoder=encoder, frozen_only=True, image_cache=IMAGE_CACHE,
                             device=DEVICE, seed=EXT_SEEDS[0])
        screen_runs[encoder] = [result["out_dir"]]


### 8.3 View the screening scores

These are validation results for the simpler classifiers, not the fully trained A-F models.


In [ ]:
if screen_runs:
    display(summary(collect(screen_runs), split="val"))
else:
    print("Enable RUN_ENCODER_SCREEN to compute encoder-screening results.")


### 8.4 Compare clinical data alone with logistic regression

This optional comparison uses a simpler clinical-data classifier. F is still the neural-network model in the main seven experiments.


In [ ]:
completed_tab = sorted(folder for folder in RUN_DIR.glob("ext_tab_logreg_*")
                       if (folder / "metrics.csv").is_file())
if completed_tab:
    display(pd.read_csv(completed_tab[-1] / "metrics.csv"))
elif RUN_TAB_BASELINE:
    from longitudinal_ext.tabular_baseline import run_tabular_baseline
    folder = RUN_DIR / datetime.now().strftime("ext_tab_logreg_%Y%m%d_%H%M%S")
    result = run_tabular_baseline(PAIR_CSV, TAB_CSV, folder, seed=EXT_SEEDS[0])
    display(pd.DataFrame(result["metrics"]))
else:
    print("Enable RUN_TAB_BASELINE to compute the tabular logistic-regression baseline.")


## 9. Optional: extract source data on a new machine
The data on this machine are already extracted. These last two cells are needed only when the source files are still inside ZIP archives. Use a dry run first to list the files without extracting them.


### 9.1 Extract the required X-ray images

Run this before the data checks in Section 2 if image files are missing. Leave `RUN_EXTRACT_DICOMS=False` when images are already available.


In [ ]:
RUN_EXTRACT_DICOMS = False
EXTRACT_DRY_RUN = True
if RUN_EXTRACT_DICOMS:
    from local_data_archives import extract_needed_dicoms
    extract_needed_dicoms(
        BASE_PAIR_CSV, [DATA_ROOT / "MIMIC-CXR" / "BN5212_MIMIC-CXR.zip"],
        REPORTS_ROOT, dry_run=EXTRACT_DRY_RUN,
        extra_manifest=LABELLING_DIR / "backbone_image_manifest.csv",
    )


### 9.2 Extract the clinical source tables

Run this before Section 3 if the clinical tables are missing. `dry_run` lists files; `extract` writes them to disk.


In [ ]:
MIMIC_IV_EXTRACT = "off"       # "off", "dry_run", or "extract"
if MIMIC_IV_EXTRACT != "off":
    from local_data_archives import extract_mimic_iv
    tables = {
        "hosp": ["patients", "admissions", "d_labitems", "labevents", "prescriptions"],
        "icu": ["icustays", "d_items", "chartevents", "inputevents", "outputevents"],
    }
    extract_mimic_iv(DATA_ROOT / "MIMIC-IV" / "MIMIC_IV.zip", MIMIC_IV_ROOT,
                     tables, dry_run=MIMIC_IV_EXTRACT == "dry_run")
